# Pandas Practice Part 2 - Task 12: Monthly Trend and Growth

**Week:** 3  
**Source:** `Pandas_Practice_Part_2.ipynb`

## Task 12: Monthly Trend and Growth
Compute total revenue per `order_month`. Then compute the **month-over-month change** in revenue (this month's total minus the previous month's total) as a new Series. (Hint: look into whether a Pandas Series method can compute the difference between consecutive values without a manual loop.)

### Prerequisites
Runs the setup and the earlier steps this task depends on (output hidden).

In [ ]:
%%capture
import pandas as pd
import numpy as np
import os

os.makedirs("sample_data", exist_ok=True)

orders_csv_text = """order_id,customer,city,product,category,quantity,unit_price,order_date
1,Ali,Lahore,Laptop,Electronics,1,1200,2024-01-05
2,Sara,Karachi,Mouse,Electronics,3,25,2024-01-06
3,Ahmed,Lahore,Desk,Furniture,1,300,2024-01-15
4,Ayesha,Islamabad,Chair,Furniture,4,150,2024-01-20
5,Ali,Lahore,Monitor,Electronics,2,220,2024-02-02
6,Bilal,Karachi,Laptop,Electronics,1,1200,2024-02-10
7,Sara,Karachi,Desk,Furniture,2,300,2024-02-14
8,Zainab,Islamabad,Chair,Furniture,3,150,2024-02-25
9,Ahmed,Lahore,Monitor,Electronics,1,220,2024-03-01
10,Ayesha,Islamabad,Mouse,Electronics,5,25,2024-03-04
11,Ali,Lahore,Chair,Furniture,2,150,2024-03-08
12,Bilal,Karachi,Monitor,Electronics,2,220,2024-03-18
"""

with open("sample_data/orders.csv", "w") as file:
    file.write(orders_csv_text)

print("orders.csv is ready.")
print(pd.__version__)

orders = pd.read_csv("sample_data/orders.csv")
orders["order_date"] = pd.to_datetime(orders["order_date"])
orders["revenue"] = orders["quantity"] * orders["unit_price"]
print(orders.dtypes)
print(orders.head())

print(orders.groupby("category")["revenue"].sum())

print(orders.groupby("city")["revenue"].agg(["sum", "mean"]))

orders["product_upper"] = orders["product"].str.upper()
print(orders[["product", "product_upper"]].head())

print("Duplicate rows in orders:", orders.duplicated().sum())

demo = pd.DataFrame({"id": [1, 2, 2, 3], "name": ["A", "B", "B", "C"]})   # row 2 is duplicated
print("\nBefore:\n", demo)
print("\nAfter drop_duplicates():\n", demo.drop_duplicates())

city_summary = orders.groupby("city").agg(
    total_orders=("order_id", "count"),
    total_revenue=("revenue", "sum"),
)
print(city_summary)

city_region = pd.DataFrame({
    "city": ["Lahore", "Karachi", "Islamabad"],
    "region": ["Punjab", "Sindh", "Federal"],
})

# how="left": keep every order from `orders` even if a city had no region entry
# (a left join never drops orders, so revenue totals stay correct).
orders_region = orders.merge(city_region, on="city", how="left")
print(orders_region[["order_id", "city", "region"]].head())

pivot = orders.pivot_table(values="revenue", index="category", columns="city", aggfunc="sum", fill_value=0)
print(pivot)

def label_order(qty):
    return "Bulk" if qty >= 3 else "Regular"

orders["order_type"] = orders["quantity"].apply(label_order)
print(orders[["order_id", "quantity", "order_type"]])

orders["order_month"] = orders["order_date"].dt.month
orders["order_weekday"] = orders["order_date"].dt.day_name()
print(orders[["order_date", "order_month", "order_weekday"]].head())

weekday_revenue = orders.groupby("order_weekday")["revenue"].sum()
print("\nRevenue by weekday:\n", weekday_revenue.sort_values(ascending=False))
print("Highest-revenue weekday:", weekday_revenue.idxmax())

cust = orders.groupby("customer").agg(
    total_orders=("order_id", "count"),
    total_spent=("revenue", "sum"),
    average_order_value=("revenue", "mean"),
)

cat_counts = orders.groupby(["customer", "category"]).size().reset_index(name="n")
cat_counts = cat_counts.sort_values(["customer", "n", "category"], ascending=[True, False, True])
cust["favorite_category"] = cat_counts.drop_duplicates("customer").set_index("customer")["category"]
print(cust)

### Solution

In [ ]:
monthly_revenue = orders.groupby("order_month")["revenue"].sum()
mom_change = monthly_revenue.diff()          # this month minus previous month
print(pd.DataFrame({"revenue": monthly_revenue, "mom_change": mom_change}))